# Implementation of YOLOv12 trained LiDar dataset. focus on reflective properties with use of hyperparameters

Every snowpole has a reflective piece on it, the training takes this in to concideration, focusing on brigth small areas in a dark environment. Placed on in the middel of every pole

In [ ]:
!git clone https://github.com/sunsmarterjie/yolov12
%cd yolov12
%pip install roboflow supervision flash-attn --upgrade -q
%pip install -r requirements.txt
%pip install -e .
%pip install --upgrade flash-attn

# Not in tutorial but necesarry:
%pip install huggingface_hub ultralytics

In [ ]:
dataset_path_lidar = "/home/edvarsa/pole_detection/lidar_1/"

In [ ]:
from ultralytics import YOLO

#Initialize YOLOv12 model
model = YOLO('yolov12s.yaml')


results = model.train(
    data=f'{dataset_path_lidar}data.yaml',
    epochs=100,
    imgsz=1024,
    batch=16,              # tune based on your GPU memory
    lr0=0.01,              # initial learning rate
    lrf=0.01,              # final learning rate (lr0 * lrf)
    momentum=0.937,        # SGD momentum
    weight_decay=0.0005,   # regularization
    warmup_epochs=3.0,     # gradual LR warm-up
    warmup_bias_lr=0.1,    
    box=0.05,              # box loss gain
    cls=0.5,               # classification loss gain
    dfl=1.5,               # distribution focal loss gain (for objectness)
    hsv_h=0.015,           # color augmentation (useful even for false RGBs)
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,           # disable rotation (LiDAR view may be orientation-specific)
    translate=0.1,         
    scale=0.5,
    shear=0.0,             # avoid shear for this kind of data
    flipud=0.0,            # disable vertical flip
    fliplr=0.5,            # horizontal flip (if poles can appear on both sides)
    mosaic=1.0,            # strong augmentation
    mixup=0.1,
    copy_paste=0.0,
    patience=20,           # early stopping patience
    device=0,
    workers=4,             # adjust depending on CPU
    project='runs',
    name='yolov12_pole_detection',
    exist_ok=True
)


In [ ]:
# Create a dictionary with hyperparameters
hyperparameters = {
    'lr0': 0.01,              # initial learning rate
    'lrf': 0.01,              # final learning rate
    'momentum': 0.937,        # SGD momentum
    'weight_decay': 0.0005,   # regularization
    'warmup_epochs': 3.0,     # warmup epochs
    'warmup_bias_lr': 0.1,    # warmup bias lr
    'box': 0.05,              # box loss gain
    'cls': 0.5,               # classification loss gain
    'dfl': 1.5,               # distribution focal loss gain
    'hsv_h': 0.015,           # HSV augmentation
    'hsv_s': 0.7,
    'hsv_v': 0.4,
    'degrees': 0.0,           # rotation
    'translate': 0.1,         # translation
    'scale': 0.5,             # scale
    'shear': 0.0,             # shear
    'flipud': 0.0,            # flip up-down
    'fliplr': 0.5,            # flip left-right
    'mosaic': 1.0,            # mosaic augmentation
    'mixup': 0.1,             # mixup augmentation
    'copy_paste': 0.0         # copy-paste augmentation
}

# First, save hyperparameters with absolute path
import os
import yaml

# Create the hyperparameters directory if it doesn't exist
hyp_dir = os.path.join(os.path.dirname(dataset_path_lidar), 'hyperparameters')
os.makedirs(hyp_dir, exist_ok=True)

# Full path to hyp.yaml
hyp_path = os.path.join(hyp_dir, 'hyp.yaml')

# Save hyperparameters to yaml file
with open(hyp_path, 'w') as f:
    yaml.dump(hyperparameters, f)

#### evolution training code with genetic algortithm

In [ ]:
from ultralytics import YOLO

# Initialize model
model = YOLO('yolov12s.yaml')

# Run training with hyperparameters
results = model.train(
    data=f'{dataset_path_lidar}data.yaml',
    cfg=hyp_path,           # Changed from hyp to cfg for hyperparameter config
    epochs=50,             
    imgsz=1024,           
    batch=16,             
    device=0,             
    workers=4,            
    project='evolve_runs', 
    name='evolve_lidar_pole',
    exist_ok=True,
    cache=True,
    optimizer='SGD',        # Explicitly specify optimizer
    lr0=0.01,              # Learning rate parameters are passed directly
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005
)

#### run eveolved training 

In [ ]:
# Load best evolved hyperparameters
best_hyp = "/home/edvarsa/pole_detection/TDT4265-Snow-pole-detection/yolov12_our_files/yolov12/yolov12/hyp.yaml"

# Train with evolved hyperparameters
model = YOLO('yolov12s.yaml')
model.train(
    data=f'{dataset_path_lidar}data.yaml',
    cfg=best_hyp,
    epochs=100,
    imgsz=1024,
    project='runs',
    name='yolov12_pole_detection_evolved'
)

#### To make testfolder for leaderboard

In [1]:
from ultralytics import YOLO

model = YOLO("/home/edvarsa/pole_detection/TDT4265-Snow-pole-detection/yolov12_our_files/yolov12/map95_optimization/refined_map95/weights/best.pt")

model.predict(
    source="/home/edvarsa/pole_detection/lidar_1/test/images",
    project="SAVE_FOLDER",
    name="SUB_SAVE_FOLDER",
    save_txt=True,
    save_conf=True # <--- This adds the probability of each predicted box
    )

/home/edvarsa/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


OSError: [Errno 122] Disk quota exceeded: 'SAVE_FOLDER'